## ESZA019 – Visao Computacional 2026.2
## Laboratório 2 –  Extração  de  Características Features

### Integrantes: 

- Eduarda Alexandre de Salles RA: 11202320551

- Gustavo de Paula Souza RA: 11202130568

- Paloma Cristina Santana RA:11201921396


#### Data de realização: 03/06/2026

#### Data de publicação: 



## 1. INTRODUÇÃO

Esta seção aborda os fundamentos teóricos necessários para a compreensão e implementação dos experimentos práticos de detecção, descrição e casamento de características locais em imagens e fluxos de vídeo.

### 1.1. Entendendo o Conceito de Features

As características (*features*) são pontos de interesse contidos numa imagem que se destacam geometricamente ou texturalmente em relação à sua viagem imediata. Elas representam descontinuidades de padrões visuais locais, tais como cantos, bordas pronunciadas ou manchas de alta intensidade. Estruturalmente, funcionam como âncoras ou assinaturas matemáticas estáveis. Isso permite a algoritmos de computação visual correlacionar e rastrear o mesmo objeto tridimensional ou elemento de cenário mesmo quando submetido a mudanças drásticas de perspectiva espacial, variações nas condições de iluminação ambiental ou transformações de escala geométrica.

### 1.2. Detector de Cantos de Harris (Harris Corner Detector)

O método de Harris formula a detecção de cantos avaliando a mudança matemática de intensidade local produzida pelo deslocamento infinitesimal de uma janela local ($[u, v]$) sobre uma região da imagem ($I$). Matematicamente, a variação de intensidade $E(u,v)$ é expressa por:

$$E(u,v) = \sum_{x,y} w(x,y)[I(x+u, y+v) - I(x,y)]^2$$

Onde $w(x,y)$ define a função de janela (geralmente uma Gaussiana). Através da expansão em Série de Taylor, a equação pode ser simplificada numa forma matricial:

$$E(u,v) \approx \begin{bmatrix} u & v \end{bmatrix} M \begin{bmatrix} u \\ v \end{bmatrix}$$

A Matriz de Estrutura Local ($M$) é dada pela covariância dos gradientes de imagem:

$$M = \sum_{x,y} w(x,y) \begin{bmatrix} I_x^2 & I_x I_y \\ I_x I_y & I_y^2 \end{bmatrix}$$

O algoritmo computa uma função de resposta ao canto ($R$), dada por:

$$R = \text{det}(M) - k \cdot (\text{trace}(M))^2$$

Onde $\text{det}(M) = \lambda_1 \lambda_2$, $\text{trace}(M) = \lambda_1 + \lambda_2$, e $k$ é uma constante empírica (comumente entre $0.04$ e $0.06$). Se os autovalores $\lambda_1$ e $\lambda_2$ forem ambos grandes e positivos, $R$ assume um valor alto, caracterizando matematicamente a presença de um **canto**, que representa uma variação brusca em todas as direções de deslocamento.

### 1.3. Detector de Shi-Tomasi (Good Features to Track)

Proposto como uma otimização direta do detector de Harris, o algoritmo de Shi-Tomasi introduz uma modificação na função de pontuação de canto ($R$). Ao analisar a estabilidade matemática para rastreamento de fluxos ópticos, os autores demonstraram que a classificação de um canto é mais robusta quando baseada unicamente no limitante inferior dos autovalores da matriz $M$. A função simplificada é definida por:

$$R = \min(\lambda_1, \lambda_2)$$

A região avaliada é consolidada como um canto válido se, e somente se, o menor autovalor for estritamente maior que um limiar mínimo pré-estipulado ($\lambda_{\text{min}}$). Esse critério garante uma distribuição homogênea e estável de pontos-chave sob deformações afins, tornando-o ideal para algoritmos de rastreamento temporal.

### 1.4. SIFT (Scale-Invariant Feature Transform)

Diferente dos métodos anteriores que operam de forma estática numa resolução fixa, o SIFT extrai pontos de interesse que apresentam invariância robusta frente a variações de escala e rotação geométrica. O processo divide-se em quatro etapas principais:

1. **Detecção de Extremos no Espaço de Escala:** A imagem é sucessivamente convoluída com núcleos Gaussianos em diferentes escalas e subamostrada (oitavas). A subtração entre escalas consecutivas gera o espaço de Diferença de Gaussianas (DoG). Os pontos-chave candidatos são os extremos locais (máximos e mínimos) detectados comparando cada pixel com os seus 26 vizinhos tridimensionais no espaço e na escala.
2. **Localização do Ponto-Chave:** Filtram-se os pontos de baixo contraste ou localizados ao longo de bordas instáveis através da análise da matriz Hessiana local.
3. **Atribuição de Orientação:** Calcula-se a magnitude e a direção do gradiente de intensidade na mudança de cada ponto-chave. Cria-se um histograma de orientação para atribuir uma direção dominante, garantindo a **invariância à rotação**.
4. **Descritor de Ponto-Chave:** A vizinhança de $16 \times 16$ pixels ao redor do ponto é subdividida em sub-regiões de $4 \times 4$. Para cada sub-região, calcula-se um histograma de orientações com 8 bins de direção. O resultado consolida um vetor descritor altamente discriminativo de **128 dimensões** ($4 \times 4 \times 8$), normalizado para mitigar variações de iluminação.


## 2. PROCEDIMENTOS EXPERIMENTAIS

Esta seção documenta a lógica operacional adotada no desenvolvimento dos programas em Python e OpenCV para a validação dos conceitos de casamento de características e homografia.

### 2.1. PARTE 2 (A): Feature Matching e Homografia Estática

Para a primeira atividade prática, foi desenvolvido um script estruturado para carregar duas imagens previamente registradas no sistema de arquivos local. A imagem `foto.png` representou o objeto de consulta isolado, enquanto a imagem `video1.png` representou o cenário complexo onde o objeto se encontrava em uma posição distinta.

O procedimento experimental seguiu o seguinte fluxo lógico:

* **Detecção:** O extrator SIFT foi instanciado para mapear os pontos de interesse e computar os vetores descritores de 128 dimensões para ambas as imagens.
* **Pareamento:** Os descritores foram processados pelo combinador baseado no algoritmo FLANN (*Fast Library for Approximate Nearest Neighbors*), configurado com árvores KD-Tree indexadas para otimizar o tempo de busca em alta dimensão.
* **Filtragem:** Aplicou-se o Teste de Razão de Lowe com fator limite de $0.7$ para rejeitar correspondências ambíguas ou falsos-positivos.
* **Mapeamento Geométrico:** Através da função de busca de homografia associada ao algoritmo RANSAC, calculou-se a matriz de transformação perspectiva estável. Caso o número de pontos válidos superasse o limiar mínimo configurado para 20 correspondências, o algoritmo projetava as coordenadas das quinas do objeto original sobre a cena, delimitando sua localização exata por meio de um polígono colorido.

In [3]:
import numpy as np
import cv2 as cv
from matplotlib import pyplot as plt

MIN_MATCH_COUNT = 20

img1 = cv.imread('foto.png', cv.IMREAD_GRAYSCALE)          # queryImage
img2 = cv.imread('video1.png', cv.IMREAD_GRAYSCALE) # trainImage

# Initiate SIFT detector
sift = cv.SIFT_create()

# find the keypoints and descriptors with SIFT
kp1, des1 = sift.detectAndCompute(img1,None)
kp2, des2 = sift.detectAndCompute(img2,None)

FLANN_INDEX_KDTREE = 1
index_params = dict(algorithm = FLANN_INDEX_KDTREE, trees = 5)
search_params = dict(checks = 50)

flann = cv.FlannBasedMatcher(index_params, search_params)

matches = flann.knnMatch(des1,des2,k=2)

# store all the good matches as per Lowe's ratio test.
good = []
for m,n in matches:
    if m.distance < 0.7*n.distance:
        good.append(m)
    if len(good)>MIN_MATCH_COUNT:
        src_pts = np.float32([ kp1[m.queryIdx].pt for m in good ]).reshape(-1,1,2)
        dst_pts = np.float32([ kp2[m.trainIdx].pt for m in good ]).reshape(-1,1,2)

        M, mask = cv.findHomography(src_pts, dst_pts, cv.RANSAC,5.0)
        matchesMask = mask.ravel().tolist()

        h,w = img1.shape
        pts = np.float32([ [0,0],[0,h-1],[w-1,h-1],[w-1,0] ]).reshape(-1,1,2)
        dst = cv.perspectiveTransform(pts,M)

        img2 = cv.polylines(img2,[np.int32(dst)],True,255,3, cv.LINE_AA)
    else:
        print( "Not enough matches are found - {}/{}".format(len(good), MIN_MATCH_COUNT) )
        matchesMask = None

draw_params = dict(matchColor = (0,255,0), # draw matches in green color
                   singlePointColor = None,
                   matchesMask = matchesMask, # draw only inliers
                   flags = 2)

img3 = cv.drawMatches(img1,kp1,img2,kp2,good,None,**draw_params)

# plt.imshow(img3, 'gray'),plt.show()
cv.imwrite('sfit.png',img3)

Not enough matches are found - 0/20
Not enough matches are found - 0/20
Not enough matches are found - 0/20
Not enough matches are found - 0/20
Not enough matches are found - 0/20
Not enough matches are found - 0/20
Not enough matches are found - 0/20
Not enough matches are found - 0/20
Not enough matches are found - 0/20
Not enough matches are found - 0/20
Not enough matches are found - 0/20
Not enough matches are found - 0/20
Not enough matches are found - 0/20
Not enough matches are found - 0/20
Not enough matches are found - 0/20
Not enough matches are found - 0/20
Not enough matches are found - 0/20
Not enough matches are found - 0/20
Not enough matches are found - 0/20
Not enough matches are found - 0/20
Not enough matches are found - 0/20
Not enough matches are found - 0/20
Not enough matches are found - 0/20
Not enough matches are found - 0/20
Not enough matches are found - 0/20
Not enough matches are found - 0/20
Not enough matches are found - 0/20
Not enough matches are found

True


### 2.2. PARTE 2 (B): Feature Matching Estereoscópico Dinâmico

Para expandir o experimento analítico para o domínio temporal, modificou-se a arquitetura do programa anterior para dar suporte ao processamento paralelo e simultâneo de hardware. Em vez de arquivos estáticos em disco, o pipeline passou a capturar os fluxos de vídeo em tempo real gerados por duas webcams configuradas em um arranjo estereoscópico calibrado.

A dinâmica adaptada operou sob as seguintes restrições:

* Implementação de um laço de repetição contínuo acoplado aos barramentos de captura de vídeo, realizando a leitura síncrona dos quadros correspondentes à câmera esquerda e à câmera direita.
* Conversão imediata e obrigatória de cada par de quadros para a escala de cinza para compatibilização matemática com o pipeline do SIFT.
* Execução paralela da detecção de pontos-chave e extração de descritores nos fluxos esquerdo e direito.
* Realização do casamento de características via FLANN em tempo de execução, desenhando dinamicamente as linhas de correspondência válidas entre as duas telas integradas e exibindo o resultado instantaneamente em uma janela unificada de vídeo até o comando de interrupção pelo usuário.


In [ ]:
import numpy as np
import cv2 as cv

MIN_MATCH_COUNT = 10

# 1. Inicializa as duas webcams (0 e 1 geralmente são os índices das câmeras acopladas)
# Se estiver usando uma câmera estéreo de cabo único que junta os feeds, 
# pode ser necessário abrir apenas o '0' e cortar a imagem ao meio.
cap_left = cv.VideoCapture(0)
cap_right = cv.VideoCapture(1)

# Inicializa o detector SIFT
sift = cv.SIFT_create()

# Configurações do FLANN Matcher
FLANN_INDEX_KDTREE = 1
index_params = dict(algorithm=FLANN_INDEX_KDTREE, trees=5)
search_params = dict(checks=50)
flann = cv.FlannBasedMatcher(index_params, search_params)

print("Pressione 'q' para sair.")

while True:
    # 2. Captura o frame de ambas as câmeras
    ret_l, frame_l = cap_left.read()
    ret_r, frame_r = cap_right.read()

    # Se falhar em ler qualquer uma das câmeras, interrompe o loop
    if not ret_l or not ret_r:
        print("Erro ao capturar frames das webcams.")
        break

    # Converte os frames para escala de cinza para o processamento do SIFT
    gray_l = cv.cvtColor(frame_l, cv.COLOR_BGR2GRAY)
    gray_r = cv.cvtColor(frame_r, cv.COLOR_BGR2GRAY)

    # 3. Detecta os keypoints e descritores em ambos os feeds de vídeo
    kp1, des1 = sift.detectAndCompute(gray_l, None)
    kp2, des2 = sift.detectAndCompute(gray_r, None)

    # Garante que ambos os frames possuem descritores antes de tentar o match
    if des1 is not None and des2 is not None:
        matches = flann.knnMatch(des1, des2, k=2)

        # Filtro pelo teste de razão de Lowe
        good = []
        for m, n in matches:
            if m.distance < 0.7 * n.distance:
                good.append(m)

        # 4. Se encontrar correspondências suficientes, calcula a Homografia
        if len(good) > MIN_MATCH_COUNT:
            src_pts = np.float32([kp1[m.queryIdx].pt for m in good]).reshape(-1, 1, 2)
            dst_pts = np.float32([kp2[m.trainIdx].pt for m in good]).reshape(-1, 1, 2)

            M, mask = cv.findHomography(src_pts, dst_pts, cv.RANSAC, 5.0)
            matchesMask = mask.ravel().tolist()

            # Desenha o retângulo outline no frame direito indicando o objeto correspondente
            h, w = gray_l.shape
            pts = np.float32([[0, 0], [0, h - 1], [w - 1, h - 1], [w - 1, 0]]).reshape(-1, 1, 2)
            
            if M is not None:
                dst = cv.perspectiveTransform(pts, M)
                frame_r = cv.polylines(frame_r, [np.int32(dst)], True, (0, 255, 0), 3, cv.LINE_AA)
        else:
            matchesMask = None
            
        # Parâmetros para desenhar as linhas de correspondência
        draw_params = dict(matchColor=(0, 255, 0),
                           singlePointColor=None,
                           matchesMask=matchesMask,
                           flags=cv.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS)

        # Cria a imagem combinada lado a lado com os matches
        output_video = cv.drawMatches(frame_l, kp1, frame_r, kp2, good, None, **draw_params)
    else:
        # Caso não encontre features, apenas junta os dois frames lado a lado para não travar o vídeo
        output_video = np.hstack((frame_l, frame_r))

    # 5. Mostra o resultado em tempo real
    # cv.imshow('Estereoscopia - SIFT Matches em Video', output_video)
    cv.imwrite('resultado-estereoscopia.avi', output_video)
    

    # Condição de saída: pressionar a tecla 'q'
    # if cv.waitKey(1) & 0xFF == ord('q'):
    #     break

# Libera os recursos das câmeras e fecha as janelas
cap_left.release()
cap_right.release()
cv.destroyAllWindows()


## 3. ANÁLISE E DISCUSSÃO DOS ESTUDOS REALIZADOS

Os experimentos práticos evidenciaram as nuances operacionais e limitações dos métodos implementados:

1. **Robustez Matemática vs. Desempenho:** Conforme fundamentado, o SIFT oferece alta invariância a rotações e variações geométricas rigorosas. Contudo, a geração e o casamento de descritores de 128 dimensões impõem um elevado custo computacional. Nos testes de execução em vídeo em tempo real, observou-se uma queda na taxa de quadros por segundo (FPS) quando a densidade de *keypoints* no cenário aumentava substancialmente, limitando o seu uso direto em sistemas embarcados restritos sem aceleração por hardware.
2. **Filtragem de Casamentos Falsos (Inliers):** O teste de razão de Lowe fixado em $0.7$ mostrou-se imprescindível. Sem ele, ambiguidades geradas por texturas repetitivas ou ruído de alta frequência produzem falsas associações geométricas. O método RANSAC atuou de forma eficiente, isolando estes ruídos matemáticos e garantindo uma estimativa correta da homografia projetiva planar.

### 3.1. Aplicação Prática Proposta para o Trabalho Final da Disciplina

As técnicas estudadas neste laboratório fornecem a base algorítmica essencial para a modelagem do trabalho final da equipe, focado em **reconstrução tridimensional estereoscópica e odometria visual**.

A extração de pontos-chave estáveis através do SIFT permite estabelecer correspondências homólogas robustas entre a imagem da câmera esquerda e a da câmera direita. Dispondo dos parâmetros intrínsecos e extrínsecos da câmera estéreo previamente calibrada, calcula-se o mapa de disparidade. A disparidade calculada pela diferença das coordenadas horizontais ($d = x_l - x_r$) possibilita extrair geometricamente a informação de profundidade absoluta ($Z$) dos objetos através do modelo de triangulação estéreo clássico:

$$Z = \frac{B \cdot f}{d}$$

Onde $B$ representa a distância de linha de base (*baseline*) entre os eixos ópticos das webcams e $f$ denota a distância focal efetiva obtida na calibração. Paralelamente, o rastreamento temporal dessas características correlacionadas permite estimar a matriz essencial entre quadros consecutivos, viabilizando o cálculo da trajetória cinemática do sistema robótico no espaço por meio de odometria visual.


## 4. CONCLUSÕES

O laboratório possibilitou fixar de forma sólida os conceitos matemáticos por trás da representação de imagens por meio de características locais de baixo nível. A transição prática de códigos que operam sobre matrizes estáticas para pipelines dinâmicos de vídeo em tempo real evidenciou desafios práticos de engenharia e visão computacional. Lidar com problemas reais como a latência de sincronismo do barramento USB, flutuação na taxa de quadros (FPS), manipulação concorrente de múltiplos fluxos de hardware e tratamento de oclusões espaciais são competências críticas absorvidas pelo grupo, fundamentais para o desenvolvimento do projeto final da disciplina.


## 5. REFERÊNCIAS CONSULTADAS 

BRADSKI, Gary; KAEHLER, Adrian. **Learning OpenCV**: Computer vision with the OpenCV library. Sebastopol: O'Reilly Media, Inc., 2008.

LOWE, David G. Distinctive image features from scale-invariant keypoints. **International Journal of Computer Vision**, v. 60, n. 2, p. 91-110, 2004.

OPENCV. **Feature Detection and Description**. OpenCV 4.x Documentation. Disponível em: [https://docs.opencv.org/](https://www.google.com/search?q=https://docs.opencv.org/). Acesso em: 07 jun. 2026.

OPENCV. **Introduction to SIFT (Scale-Invariant Feature Transform)**. Disponível em: [https://docs.opencv.org/4.x/da/df5/tutorial_py_sift_intro.html](https://www.google.com/search?q=https://docs.opencv.org/4.x/da/df5/tutorial_py_sift_intro.html). Acesso em: 07 jun. 2026.

OPENCV. **Harris Corner Detection**. Disponível em: [https://docs.opencv.org/4.x/dc/d0d/tutorial_py_features_harris.html](https://www.google.com/search?q=https://docs.opencv.org/4.x/dc/d0d/tutorial_py_features_harris.html). Acesso em: 07 jun. 2026.

OPENCV. **Shi-Tomasi Corner Detector & Good Features to Track**. Disponível em: [https://docs.opencv.org/4.x/d4/d8c/tutorial_py_shi_tomasi.html](https://www.google.com/search?q=https://docs.opencv.org/4.x/d4/d8c/tutorial_py_shi_tomasi.html). Acesso em: 07 jun. 2026.

OPENCV. **Feature Matching + Homography to find Objects**. Disponível em: [https://docs.opencv.org/4.x/d1/de6/tutorial_py_homography.html](https://www.google.com/search?q=https://docs.opencv.org/4.x/d1/de6/tutorial_py_homography.html). Acesso em: 07 jun. 2026.